In [ ]:
import os

os.environ["PFI_RELEASE_REPO_REF"] = (
    "ebe0135dc4f25e563fa3ddc92ff22b4e53a1f5b1"
)

os.environ["PFI_REPO_ROOT"] = (
    "/content/PFI_MVPTest_Enzo_AImodule"
)

# Dry-run obligatorio: no publicar.
os.environ["RUN_GCS_RELEASE_PUBLISH"] = "0"
os.environ.pop(
    "CONFIRM_GCS_RELEASE_PUBLISH",
    None,
)

print({
    "PFI_RELEASE_REPO_REF":
        os.environ["PFI_RELEASE_REPO_REF"],
    "RUN_GCS_RELEASE_PUBLISH":
        os.environ["RUN_GCS_RELEASE_PUBLISH"],
    "CONFIRM_GCS_RELEASE_PUBLISH":
        os.environ.get(
            "CONFIRM_GCS_RELEASE_PUBLISH"
        ),
})

# Publicación segura del modelo sagital final v1

Este notebook prepara, valida, versiona y publica de forma controlada la release `sagittal_spider_final_v1` del modelo sagital definitivo. No entrena modelos, no modifica el pipeline de inferencia y no toca el modelo axial.

Por defecto funciona como dry-run: monta Drive, valida artefactos fuente, construye staging local, genera manifest/model card/release manifest y muestra el plan. La escritura remota solo queda disponible detrás del gate explícito `RUN_GCS_RELEASE_PUBLISH=1` y `CONFIRM_GCS_RELEASE_PUBLISH=PUBLISH-SPIDER-SAGITTAL-FINAL-V1-6013e160`.

## Advertencias académicas y alcance

El modelo es académico y asistivo. Requiere revisión profesional humana, no constituye validación clínica, no reemplaza criterio profesional y no debe producir diagnóstico ni recomendaciones terapéuticas. Este notebook solo publica artifacts bajo `gs://pfi-rm-lumbar-artifacts-2026-ef/models/releases/sagittal_spider_final_v1/`.

In [ ]:
from __future__ import annotations

import importlib.util
import json
import math
import os
import re
import shutil
import subprocess
import sys
import tempfile
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

REQUIRED_MODULES = {
    "google.cloud.storage": "google-cloud-storage",
    "pandas": "pandas",
    "torch": "torch",
    "nbformat": "nbformat",
}
missing = [pkg for module, pkg in REQUIRED_MODULES.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *sorted(set(missing))])

import hashlib
import nbformat
import pandas as pd
import torch
from google.cloud import storage

## Configuración inmutable de release

In [ ]:
@dataclass(frozen=True)
class ReleaseConfig:
    project_id: str = "pfi-asplanatti-fabrello-v1"
    bucket_name: str = "pfi-rm-lumbar-artifacts-2026-ef"
    release_id: str = "sagittal_spider_final_v1"
    destination_prefix: str = "models/releases/sagittal_spider_final_v1"
    source_output_dir: Path = Path("/content/drive/MyDrive/PFI_MVP/results/GCS_spider_final_training_v1")
    staging_dir: Path = Path("/content/drive/MyDrive/PFI_MVP/results/GCS_spider_final_training_v1/release_packages/sagittal_spider_final_v1")
    source_model_filename: str = "sagittal_spider_final_v1.pt"
    published_model_filename: str = "sagittal_spider_multiclass_final_best.pt"
    expected_training_repository_commit: str = "6013e160f45c9263fd4ae50e864ceb37245323e2"
    expected_architecture_sha256: str = "d83f735cca9cbefc0e65dd8863466f4a528f205f3d674ebb73c49d68f8687c90"
    expected_manifest_sha256: str = "fa54c89a278d850021c0f91c0a27b3b5211c86301c9e4f125d75d517f39b793b"
    expected_training_index_sha256: str = "2720b7218c92870f6f0a000b57111ed36b5cf3b78c716f244f427ca7fee4a4ba"
    expected_quality_gate: float = 0.70
    expected_classes: tuple[str, ...] = ("background", "vertebra_group", "canal", "disc_group")
    expected_target_size: tuple[int, int] = (256, 256)
    expected_base_channels: int = 16
    expected_num_classes: int = 4
    expected_patient_counts: dict[str, int] | None = None
    expected_slice_counts: dict[str, int] | None = None
    overwrite_allowed: bool = False
    human_review_required: bool = True
    not_clinical_diagnosis: bool = True

    def __post_init__(self) -> None:
        object.__setattr__(self, "expected_patient_counts", self.expected_patient_counts or {"train": 152, "val": 33, "test": 33})
        object.__setattr__(self, "expected_slice_counts", self.expected_slice_counts or {"train": 5271, "val": 1174, "test": 1218})

CFG = ReleaseConfig()
RELEASE_GS_URI = f"gs://{CFG.bucket_name}/{CFG.destination_prefix}/"
PUBLISH_TOKEN = "PUBLISH-SPIDER-SAGITTAL-FINAL-V1-6013e160"
RUN_GCS_RELEASE_PUBLISH = os.getenv("RUN_GCS_RELEASE_PUBLISH", "0") == "1"
CONFIRM_GCS_RELEASE_PUBLISH = os.getenv("CONFIRM_GCS_RELEASE_PUBLISH", "")
gcs_read_operations = 0
gcs_write_operations = 0
objects_uploaded = 0
objects_existing_verified = 0
objects_remote_verified = 0
bytes_uploaded = 0

assert CFG.destination_prefix == "models/releases/sagittal_spider_final_v1"
assert CFG.destination_prefix.strip("/") == CFG.destination_prefix
assert ".." not in CFG.destination_prefix.split("/")
assert CFG.overwrite_allowed is False
assert CFG.human_review_required is True
assert CFG.not_clinical_diagnosis is True

## Utilidades locales

In [ ]:
def sha256_stream(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as fh:
        for chunk in iter(lambda: fh.read(1024 * 1024), b""):
            getattr(digest, "update")(chunk)
    return digest.hexdigest()

def atomic_write_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, tmp_name = tempfile.mkstemp(prefix=path.name, suffix=".tmp", dir=str(path.parent))
    with os.fdopen(fd, "w", encoding="utf-8") as fh:
        fh.write(text)
    os.replace(tmp_name, path)

def atomic_write_json(path: Path, payload: dict[str, Any]) -> None:
    text = json.dumps(payload, indent=2, sort_keys=True, allow_nan=False, ensure_ascii=False) + "\n"
    atomic_write_text(path, text)

def copy_atomic(source: Path, destination: Path) -> str:
    if not source.exists() or not source.is_file() or source.stat().st_size <= 0:
        raise FileNotFoundError(f"Fuente invalida: {source.name}")
    destination.parent.mkdir(parents=True, exist_ok=True)
    fd, tmp_name = tempfile.mkstemp(prefix=destination.name, suffix=".tmp", dir=str(destination.parent))
    os.close(fd)
    tmp_path = Path(tmp_name)
    shutil.copy2(source, tmp_path)
    if tmp_path.stat().st_size != source.stat().st_size:
        raise RuntimeError(f"Copia local incompleta: {destination.name}")
    source_sha = sha256_stream(source)
    if sha256_stream(tmp_path) != source_sha:
        raise RuntimeError(f"SHA local no coincide: {destination.name}")
    os.replace(tmp_path, destination)
    return source_sha

def read_json(path: Path) -> dict[str, Any]:
    if not path.exists() or path.stat().st_size <= 0:
        raise FileNotFoundError(f"JSON requerido ausente o vacio: {path.name}")
    return json.loads(path.read_text(encoding="utf-8"))

def finite_number(value: Any, name: str) -> float:
    if not isinstance(value, (int, float)) or isinstance(value, bool) or not math.isfinite(float(value)):
        raise ValueError(f"{name} no es numerico finito")
    return float(value)

def close_enough(actual: Any, expected: float, name: str) -> None:
    if not math.isclose(finite_number(actual, name), expected, rel_tol=1e-9, abs_tol=1e-9):
        raise ValueError(f"{name} no coincide")

def content_type_for(path: Path) -> str:
    suffix = path.suffix.lower()
    if suffix == ".json":
        return "application/json"
    if suffix == ".csv":
        return "text/csv"
    if suffix == ".png":
        return "image/png"
    if suffix == ".md":
        return "text/markdown"
    if suffix == ".pt":
        return "application/octet-stream"
    return "application/octet-stream"

def assert_hex_sha(value: str, name: str) -> None:
    if not re.fullmatch(r"[0-9a-f]{64}", str(value)):
        raise ValueError(f"{name} no es SHA-256 hex valido")

## Montaje y verificación de Drive

In [ ]:
def running_in_colab() -> bool:
    try:
        import google.colab  # type: ignore
        return True
    except Exception:
        return False

def mount_drive_if_needed() -> None:
    if not running_in_colab():
        raise RuntimeError("Este publisher esta preparado para Colab con Drive montado.")
    drive_root = Path("/content/drive")
    my_drive = drive_root / "MyDrive"
    if my_drive.exists():
        return
    from google.colab import drive  # type: ignore
    if drive_root.exists() and any(drive_root.iterdir()):
        shutil.rmtree(drive_root)
    drive.mount(str(drive_root))
    if not my_drive.exists():
        raise RuntimeError("No se pudo montar Google Drive.")

mount_drive_if_needed()
if not CFG.source_output_dir.exists():
    raise FileNotFoundError("No existe el directorio fuente del entrenamiento final.")
CFG.staging_dir.mkdir(parents=True, exist_ok=True)
print(json.dumps({"source_output_dir": str(CFG.source_output_dir), "staging_dir": str(CFG.staging_dir)}, indent=2))

## Carga y validación de artefactos fuente

In [ ]:
required_sources = {
    "model": CFG.source_model_filename,
    "final_metrics": "final_test_metrics.json",
    "history_csv": "training_history.csv",
    "history_json": "training_history.json",
    "curves": "training_curves.png",
    "preview_informative": "test_predictions_preview_informative.png",
    "summary": "gcs_spider_final_training_summary.json",
    "evidence": "gcs_spider_final_training_evidence.json",
}
optional_sources = {"preview": "test_predictions_preview.png"}
source_paths = {key: CFG.source_output_dir / filename for key, filename in required_sources.items()}
for key, path in source_paths.items():
    if not path.exists() or not path.is_file() or path.stat().st_size <= 0:
        raise FileNotFoundError(f"Fuente requerida ausente o vacia: {path.name}")
for key, filename in optional_sources.items():
    path = CFG.source_output_dir / filename
    if path.exists() and path.is_file() and path.stat().st_size > 0:
        source_paths[key] = path

final_metrics = read_json(source_paths["final_metrics"])
summary = read_json(source_paths["summary"])
evidence = read_json(source_paths["evidence"])
history_json = read_json(source_paths["history_json"])
history_csv = pd.read_csv(source_paths["history_csv"])

if summary.get("FINAL_TRAINING_SUCCESS") is not True:
    raise RuntimeError("FINAL_TRAINING_SUCCESS no es true.")
if summary.get("QUALITY_GATE_PASSED") is not True:
    raise RuntimeError("QUALITY_GATE_PASSED no es true.")
if finite_number(summary.get("test_dice_macro_no_bg"), "test dice") < CFG.expected_quality_gate:
    raise RuntimeError("El Dice final no supera el gate.")
finite_number(summary.get("test_iou_macro_no_bg"), "test iou")
if summary.get("repository_commit") != CFG.expected_training_repository_commit:
    raise RuntimeError("repository_commit inesperado.")
if summary.get("architecture_sha256") != CFG.expected_architecture_sha256:
    raise RuntimeError("architecture_sha256 inesperado.")
if summary.get("manifest_sha256") != CFG.expected_manifest_sha256:
    raise RuntimeError("manifest SHA inesperado.")
if summary.get("training_index_sha256") != CFG.expected_training_index_sha256:
    raise RuntimeError("training index SHA inesperado.")
if int(summary.get("best_epoch")) != 63 or int(summary.get("epochs_completed")) != 75:
    raise RuntimeError("epochs_completed/best_epoch inesperados.")
if summary.get("patient_counts") != CFG.expected_patient_counts:
    raise RuntimeError("Conteos de pacientes inesperados.")
actual_slice_counts = summary.get("actual_slice_counts")
if actual_slice_counts != CFG.expected_slice_counts:
    raise RuntimeError("Conteos de slices reales inesperados.")

close_enough(summary.get("best_validation_dice_macro_no_bg"), 0.8992978910787764, "best validation dice")
close_enough(summary.get("test_loss"), 0.14282359443361464, "test loss")
close_enough(summary.get("test_dice_macro_no_bg"), 0.8934316063846954, "test dice")
close_enough(summary.get("test_iou_macro_no_bg"), 0.8079981902423006, "test iou")
close_enough(final_metrics.get("dice_macro_no_bg"), summary.get("test_dice_macro_no_bg"), "metrics dice")
close_enough(final_metrics.get("iou_macro_no_bg"), summary.get("test_iou_macro_no_bg"), "metrics iou")
close_enough(final_metrics.get("test_loss"), summary.get("test_loss"), "metrics test loss")
if evidence.get("manifest_sha256") != CFG.expected_manifest_sha256:
    raise RuntimeError("Evidence manifest SHA inesperado.")
if evidence.get("training_index_sha256") != CFG.expected_training_index_sha256:
    raise RuntimeError("Evidence training index SHA inesperado.")

history_rows = history_json.get("history", [])
if len(history_rows) != len(history_csv):
    raise RuntimeError("training_history.csv y json tienen longitudes distintas.")
for idx, row in enumerate(history_rows):
    csv_row = history_csv.iloc[idx].to_dict()
    if int(row["epoch"]) != int(csv_row["epoch"]):
        raise RuntimeError("History CSV/JSON discrepan en epoch.")
    for metric_name in ["train_loss", "val_loss", "val_dice_macro_no_bg", "val_iou_macro_no_bg"]:
        if not math.isclose(float(row[metric_name]), float(csv_row[metric_name]), rel_tol=1e-9, abs_tol=1e-9):
            raise RuntimeError(f"History CSV/JSON discrepan en {metric_name}.")
best_history_row = max(history_rows, key=lambda row: float(row["val_dice_macro_no_bg"]))
if int(best_history_row["epoch"]) != 63:
    raise RuntimeError("El mejor epoch reconstruido no coincide.")

model_sha256 = sha256_stream(source_paths["model"])
assert_hex_sha(model_sha256, "model_sha256")
print(json.dumps({"model_sha256": model_sha256, "best_epoch": 63, "quality_gate_passed": True}, indent=2))

## Validación estricta del checkpoint

In [ ]:
PFI_REPO_URL = "https://github.com/EnzoAA004/PFI_MVPTest_Enzo_AImodule.git"
PFI_RELEASE_REPO_REF = os.getenv("PFI_RELEASE_REPO_REF", "main")
repo_root = Path(os.getenv("PFI_REPO_ROOT", "/content/PFI_MVPTest_Enzo_AImodule"))


def run_git(args: list[str], cwd: Path | None = None, check: bool = True) -> subprocess.CompletedProcess:
    return subprocess.run(["git", *args], cwd=str(cwd) if cwd else None, text=True, capture_output=True, check=check)


def ensure_release_repo() -> Path:
    if repo_root.exists():
        inside = run_git(["rev-parse", "--show-toplevel"], repo_root, check=False)
        if inside.returncode != 0:
            raise RuntimeError("PFI_REPO_ROOT existe pero no es un repositorio git.")
        remote = run_git(["remote", "get-url", "origin"], repo_root).stdout.strip()
        if "PFI_MVPTest_Enzo_AImodule" not in remote:
            raise RuntimeError("PFI_REPO_ROOT no apunta al repositorio AI esperado.")
        status = run_git(["status", "--porcelain"], repo_root).stdout.strip()
        if status:
            raise RuntimeError("El repo AI tiene cambios locales; resolver antes de publicar.")
    else:
        repo_root.parent.mkdir(parents=True, exist_ok=True)
        run_git(["clone", PFI_REPO_URL, str(repo_root)])
    run_git(["fetch", "origin"], repo_root)
    run_git(["checkout", PFI_RELEASE_REPO_REF], repo_root)
    if re.fullmatch(r"[0-9a-f]{40}", PFI_RELEASE_REPO_REF):
        checked = run_git(["rev-parse", "HEAD"], repo_root).stdout.strip()
        if checked != PFI_RELEASE_REPO_REF:
            raise RuntimeError("Checkout del SHA publisher no coincide.")
    return repo_root


repo_root = ensure_release_repo()
ai_service_path = repo_root / "ai_service"
if str(ai_service_path) not in sys.path:
    sys.path.insert(0, str(ai_service_path))
from pfi_ai_service.model_architectures import build_checkpoint_model
from pfi_ai_service.model_manifest import validate_manifest

checkpoint = torch.load(source_paths["model"], map_location="cpu", weights_only=False)
strict_model, runtime_meta = build_checkpoint_model("sagittal_spider", checkpoint)
strict_model.eval()
with torch.no_grad():
    strict_logits = strict_model(torch.randn(1, 1, 256, 256))
output_shape = list(strict_logits.shape)
strict_load_success = output_shape == [1, 4, 256, 256] and torch.isfinite(strict_logits).all().item()
if not strict_load_success:
    raise RuntimeError("Strict load del checkpoint final fallo.")
if int(runtime_meta.get("numClasses")) != CFG.expected_num_classes:
    raise RuntimeError("numClasses runtime inesperado.")
if list(runtime_meta.get("targetSize")) != list(CFG.expected_target_size):
    raise RuntimeError("targetSize runtime inesperado.")
if int(runtime_meta.get("baseChannels")) != CFG.expected_base_channels:
    raise RuntimeError("baseChannels runtime inesperado.")
if checkpoint.get("source_repository_commit") and checkpoint.get("source_repository_commit") != CFG.expected_training_repository_commit:
    raise RuntimeError("Checkpoint con repository_commit inesperado.")
if checkpoint.get("source_model_architecture_sha256") and checkpoint.get("source_model_architecture_sha256") != CFG.expected_architecture_sha256:
    raise RuntimeError("Checkpoint con architecture SHA inesperado.")
print(json.dumps({"strict_load_success": strict_load_success, "output_shape": output_shape, "runtime_meta": runtime_meta}, indent=2))

## Construcción local del paquete de release

In [ ]:
published_model_path = CFG.staging_dir / CFG.published_model_filename
model_sha256_from_copy = copy_atomic(source_paths["model"], published_model_path)
if model_sha256_from_copy != model_sha256:
    raise RuntimeError("SHA del alias local no coincide con el fuente.")

artifact_plan = [
    {"source_key": "model", "source": published_model_path, "filename": CFG.published_model_filename, "required": True},
    {"source_key": "final_metrics", "source": source_paths["final_metrics"], "filename": "final_test_metrics.json", "required": True},
    {"source_key": "history_csv", "source": source_paths["history_csv"], "filename": "training_history.csv", "required": True},
    {"source_key": "history_json", "source": source_paths["history_json"], "filename": "training_history.json", "required": True},
    {"source_key": "curves", "source": source_paths["curves"], "filename": "training_curves.png", "required": True},
    {"source_key": "preview_informative", "source": source_paths["preview_informative"], "filename": "test_predictions_preview_informative.png", "required": True},
    {"source_key": "summary", "source": source_paths["summary"], "filename": "gcs_spider_final_training_summary.json", "required": True},
    {"source_key": "evidence", "source": source_paths["evidence"], "filename": "gcs_spider_final_training_evidence.json", "required": True},
]
if "preview" in source_paths:
    artifact_plan.append({"source_key": "preview", "source": source_paths["preview"], "filename": "test_predictions_preview.png", "required": False})

for item in artifact_plan[1:]:
    target = CFG.staging_dir / item["filename"]
    item["sha256"] = copy_atomic(item["source"], target)
    item["path"] = target
artifact_plan[0]["sha256"] = model_sha256
artifact_plan[0]["path"] = published_model_path

excluded_fragments = ["checkpoint", ".mha", ".mhd", ".dcm", "memmap", "credential"]
for item in artifact_plan:
    lowered = item["filename"].lower()
    if any(fragment in lowered for fragment in excluded_fragments):
        raise RuntimeError(f"Archivo no permitido en release: {item['filename']}")

## Manifest runtime y model card

In [ ]:
runtime_manifest = {
    "modelKey": "sagittal_spider",
    "version": "sagittal-spider-final-v1",
    "artifactFile": CFG.published_model_filename,
    "dataset": "SPIDER sagittal lumbar MRI",
    "task": "lumbar_mri_multiclass_segmentation",
    "inputPlane": "sagittal",
    "classes": list(CFG.expected_classes),
    "metrics": {
        "dice": 0.8934316063846954,
        "iou": 0.8079981902423006,
        "scope": "final_patient_level_held_out_test_macro_no_background",
        "validationDiceMacroNoBackground": 0.8992978910787764,
        "testLoss": 0.14282359443361464,
        "qualityGateTarget": CFG.expected_quality_gate,
        "qualityGatePassed": True,
    },
    "trainingStatus": "validated_baseline",
    "sha256": model_sha256,
    "epochsCompleted": 75,
    "bestEpoch": 63,
    "reasonFinished": "early_stopping",
    "targetSize": list(CFG.expected_target_size),
    "baseChannels": CFG.expected_base_channels,
    "numClasses": CFG.expected_num_classes,
    "patientCounts": CFG.expected_patient_counts,
    "sliceCounts": actual_slice_counts,
    "manifestSha256": CFG.expected_manifest_sha256,
    "trainingIndexSha256": CFG.expected_training_index_sha256,
    "sourceRepositoryCommit": CFG.expected_training_repository_commit,
    "sourceArchitectureSha256": CFG.expected_architecture_sha256,
    "sourceTrainingNotebook": "notebooks/45_gcs_spider_final_training.ipynb",
    "sourceTrainingSummary": "gcs_spider_final_training_summary.json",
    "humanReviewRequired": CFG.human_review_required,
    "notClinicalDiagnosis": CFG.not_clinical_diagnosis,
    "clinicalValidationStatus": "not_clinically_validated",
    "warning": "Modelo académico de apoyo. Requiere revisión profesional y no debe utilizarse como sustituto del criterio clínico ni como dispositivo médico validado.",
}
runtime_manifest_path = CFG.staging_dir / f"{CFG.published_model_filename}.manifest.json"
atomic_write_json(runtime_manifest_path, runtime_manifest)
manifest_validation = validate_manifest(runtime_manifest, artifact_path=published_model_path)
if not manifest_validation.get("valid") or not manifest_validation.get("baselineReady") or manifest_validation.get("sha256Status") != "MATCH" or manifest_validation.get("validationErrors"):
    raise RuntimeError(f"Manifest runtime invalido: {manifest_validation}")

model_card = f"""# Model Card - {CFG.published_model_filename}

## Nombre y versión

- Modelo: sagittal_spider
- Versión: sagittal-spider-final-v1
- Artifact runtime: `{CFG.published_model_filename}`
- Release GCS: `{RELEASE_GS_URI}`

## Objetivo

Segmentación multiclase de RM lumbar sagital SPIDER para apoyo académico revisable. No produce diagnóstico automático ni recomendaciones terapéuticas.

## Arquitectura y formato

- Arquitectura: SagittalUNet2D
- Entrada: tensor 1x256x256
- Salida: 4 clases: background, vertebra_group, canal, disc_group
- base_channels: 16
- target_size: 256x256
- Runtime alias compatible: `{CFG.published_model_filename}`

## Dataset y split

- Dataset: SPIDER sagittal lumbar MRI
- Split por paciente: train=152, val=33, test=33
- Slices: train={actual_slice_counts['train']}, val={actual_slice_counts['val']}, test={actual_slice_counts['test']}
- Validation selecciona modelo; test se evalúa una sola vez con el mejor checkpoint.

## Entrenamiento

- Notebook fuente: notebooks/45_gcs_spider_final_training.ipynb
- Optimizer: AdamW
- Loss: CrossEntropy ponderada + Dice sin fondo
- Early stopping: patience 12
- Epochs completados: 75
- Mejor epoch: 63
- Reason finished: early_stopping

## Métricas finales

- Validation Dice macro no background: 0.8992978910787764
- Test Dice macro no background: 0.8934316063846954
- Test IoU macro no background: 0.8079981902423006
- Umbral Dice: 0.70
- Quality gate: aprobado

## Procedencia y hashes

- Model SHA-256: `{model_sha256}`
- Training repository commit: `{CFG.expected_training_repository_commit}`
- Architecture SHA-256: `{CFG.expected_architecture_sha256}`
- Dataset manifest SHA-256: `{CFG.expected_manifest_sha256}`
- Training index SHA-256: `{CFG.expected_training_index_sha256}`

## Formatos soportados por runtime

El runtime actual acepta entradas de imagen y volumen soportadas por el servicio existente. Esta release solo publica artifacts; la materialización cloud desde gs:// requiere una tarea separada controlada o una estrategia HTTPS autenticada.

## Limitaciones

- No tiene validación clínica.
- Puede sesgarse al dominio SPIDER evaluado.
- Requiere revisión profesional humana.
- No debe usarse para diagnóstico automático, seguridad clínica, eficacia clínica ni generalización fuera del conjunto evaluado.
"""
model_card_path = CFG.staging_dir / f"{CFG.published_model_filename}.modelcard.md"
atomic_write_text(model_card_path, model_card)

## Release manifest y dry-run

In [ ]:
generated_artifacts = [
    {"filename": CFG.published_model_filename, "sourceFilename": CFG.source_model_filename, "required": True, "path": published_model_path},
    {"filename": runtime_manifest_path.name, "sourceFilename": runtime_manifest_path.name, "required": True, "path": runtime_manifest_path},
    {"filename": model_card_path.name, "sourceFilename": model_card_path.name, "required": True, "path": model_card_path},
]
for item in artifact_plan[1:]:
    generated_artifacts.append({"filename": item["filename"], "sourceFilename": item["filename"], "required": item["required"], "path": item["path"]})

def artifact_entry(item: dict[str, Any]) -> dict[str, Any]:
    path = Path(item["path"])
    destination_object = f"{CFG.destination_prefix}/{item['filename']}"
    return {
        "filename": item["filename"],
        "required": bool(item["required"]),
        "sourceFilename": item["sourceFilename"],
        "sizeBytes": int(path.stat().st_size),
        "sha256": sha256_stream(path),
        "contentType": content_type_for(path),
        "destinationObject": destination_object,
        "destinationGsUri": f"gs://{CFG.bucket_name}/{destination_object}",
    }

content_artifacts = [artifact_entry(item) for item in generated_artifacts]
filenames = [item["filename"] for item in content_artifacts]
objects = [item["destinationObject"] for item in content_artifacts]
if len(filenames) != len(set(filenames)) or len(objects) != len(set(objects)):
    raise RuntimeError("Artifacts o destinos duplicados.")
for item in content_artifacts:
    assert_hex_sha(item["sha256"], item["filename"])
    if item["sizeBytes"] <= 0:
        raise RuntimeError("Artifact con tamano cero.")
    if not item["destinationObject"].startswith(CFG.destination_prefix + "/"):
        raise RuntimeError("Destino fuera del prefijo.")
    if ".." in Path(item["destinationObject"]).parts:
        raise RuntimeError("Destino con traversal.")

content_basis = [{"filename": a["filename"], "sizeBytes": a["sizeBytes"], "sha256": a["sha256"]} for a in sorted(content_artifacts, key=lambda x: x["filename"])]
release_content_sha256 = hashlib.sha256(json.dumps(content_basis, sort_keys=True, separators=(",", ":")).encode("utf-8")).hexdigest()

release_manifest = {
    "schemaVersion": 1,
    "releaseId": CFG.release_id,
    "modelKey": "sagittal_spider",
    "version": "sagittal-spider-final-v1",
    "status": "validated_release_package",
    "projectId": CFG.project_id,
    "bucket": CFG.bucket_name,
    "destinationPrefix": CFG.destination_prefix,
    "releaseGsUri": RELEASE_GS_URI,
    "sourceOutputDir": str(CFG.source_output_dir),
    "sourceRepositoryCommit": CFG.expected_training_repository_commit,
    "sourceArchitectureSha256": CFG.expected_architecture_sha256,
    "trainingManifestSha256": CFG.expected_manifest_sha256,
    "trainingIndexSha256": CFG.expected_training_index_sha256,
    "metrics": runtime_manifest["metrics"],
    "patientCounts": runtime_manifest["patientCounts"],
    "sliceCounts": actual_slice_counts,
    "qualityGate": {"target": CFG.expected_quality_gate, "passed": True},
    "humanReviewRequired": CFG.human_review_required,
    "notClinicalDiagnosis": CFG.not_clinical_diagnosis,
    "clinicalValidationStatus": "not_clinically_validated",
    "cloudRuntimeMaterializationNote": "Cloud runtime materialization from gs:// requires a separate controlled integration task or an authenticated HTTPS materialization strategy.",
    "artifacts": content_artifacts,
    "totalFiles": len(content_artifacts),
    "totalBytes": int(sum(a["sizeBytes"] for a in content_artifacts)),
    "releaseContentSha256": release_content_sha256,
}
release_manifest_path = CFG.staging_dir / "release_manifest.json"
atomic_write_json(release_manifest_path, release_manifest)
release_manifest_entry = artifact_entry({"filename": "release_manifest.json", "sourceFilename": "release_manifest.json", "required": True, "path": release_manifest_path})
publication_artifacts = [*content_artifacts, release_manifest_entry]

if sha256_stream(release_manifest_path) != release_manifest_entry["sha256"]:
    raise RuntimeError("releaseManifestEntry no coincide con el archivo final.")
if release_manifest_path.stat().st_size != release_manifest_entry["sizeBytes"]:
    raise RuntimeError("releaseManifestEntry tiene tamaño incorrecto.")
if any(item["filename"] == "release_manifest.json" for item in release_manifest["artifacts"]):
    raise RuntimeError("release_manifest.json no debe ser autorreferencial.")
if sum(item["filename"] == "release_manifest.json" for item in publication_artifacts) != 1:
    raise RuntimeError("release_manifest.json debe aparecer una vez en publication_artifacts.")


def synthetic_reproducibility_self_check() -> dict[str, Any]:
    synthetic_content = [
        {"filename": "a.bin", "sizeBytes": 10, "sha256": "0" * 64},
        {"filename": "b.json", "sizeBytes": 20, "sha256": "1" * 64},
    ]
    basis = [{"filename": item["filename"], "sizeBytes": item["sizeBytes"], "sha256": item["sha256"]} for item in sorted(synthetic_content, key=lambda item: item["filename"])]
    first = hashlib.sha256(json.dumps(basis, sort_keys=True, separators=(",", ":")).encode("utf-8")).hexdigest()
    second = hashlib.sha256(json.dumps(basis, sort_keys=True, separators=(",", ":")).encode("utf-8")).hexdigest()
    simulated_success = {"releaseId": CFG.release_id, "status": "published_and_verified", "releaseContentSha256": first, "remoteVerificationPassed": True}
    existing_release_status = "existing_release_verified" if simulated_success["releaseContentSha256"] == first and simulated_success["remoteVerificationPassed"] is True else "invalid"
    return {
        "release_manifest_not_self_referential": all(item["filename"] != "release_manifest.json" for item in synthetic_content),
        "release_content_sha256_deterministic": first == second,
        "existing_success_status": existing_release_status,
        "simulated_gcs_write_operations": 0,
    }

plan_df = pd.DataFrame([{
    "local_file": item["filename"],
    "size": item["sizeBytes"],
    "sha256": item["sha256"],
    "content_type": item["contentType"],
    "destination_uri": item["destinationGsUri"],
    "required": item["required"],
    "remote_action_plan": "verify_or_create_after_gate",
} for item in publication_artifacts])
publication_status = "dry_run_ready"
print(plan_df)
print(json.dumps({"publication_status": publication_status, "gcs_write_operations": gcs_write_operations, "release_content_sha256": release_content_sha256}, indent=2))

## Gate explícito y publicación idempotente

In [ ]:
def authenticate_for_publication():
    if not RUN_GCS_RELEASE_PUBLISH or CONFIRM_GCS_RELEASE_PUBLISH != PUBLISH_TOKEN:
        print(json.dumps({
            "expected_token": PUBLISH_TOKEN,
            "dry_run": True,
            "publication_status": "dry_run_ready",
        }, indent=2))
        return None, None
    from google.colab import auth  # type: ignore
    auth.authenticate_user()
    client = storage.Client(project=CFG.project_id)
    bucket = client.bucket(CFG.bucket_name)
    bucket.reload()
    count_gcs_read("bucket.reload")
    if bucket.name != CFG.bucket_name:
        raise RuntimeError("Bucket inesperado.")
    if getattr(bucket, "location", None) != "US-CENTRAL1":
        raise RuntimeError("Location de bucket inesperada.")
    if getattr(bucket, "storage_class", None) != "STANDARD":
        raise RuntimeError("Storage class inesperado.")
    return client, bucket


def count_gcs_read(reason: str) -> None:
    global gcs_read_operations
    gcs_read_operations += 1


def count_gcs_write(size_bytes: int) -> None:
    global gcs_write_operations, bytes_uploaded
    gcs_write_operations += 1
    bytes_uploaded += int(size_bytes)


def blob_exists_counted(blob) -> bool:
    exists = blob.exists()
    count_gcs_read("blob.exists")
    return bool(exists)


def blob_reload_counted(blob) -> None:
    blob.reload()
    count_gcs_read("blob.reload")


def download_blob_to_temp(blob) -> Path:
    with tempfile.NamedTemporaryFile(delete=False) as tmp:
        tmp_path = Path(tmp.name)
    blob.download_to_filename(str(tmp_path))
    count_gcs_read("blob.download_to_filename")
    return tmp_path


def read_remote_json_and_sha256(blob) -> tuple[dict[str, Any], str]:
    tmp_path = download_blob_to_temp(blob)
    try:
        payload = json.loads(tmp_path.read_text(encoding="utf-8"))
        return payload, sha256_stream(tmp_path)
    finally:
        tmp_path.unlink(missing_ok=True)


def read_remote_json(blob) -> dict[str, Any]:
    payload, _ = read_remote_json_and_sha256(blob)
    return payload


def read_remote_sha256(blob) -> str:
    tmp_path = download_blob_to_temp(blob)
    try:
        return sha256_stream(tmp_path)
    finally:
        tmp_path.unlink(missing_ok=True)


def verify_remote_artifact(bucket, item: dict[str, Any]) -> dict[str, Any]:
    global objects_existing_verified, objects_remote_verified
    blob = bucket.blob(item["destinationObject"])
    if not blob_exists_counted(blob):
        raise RuntimeError(f"Objeto remoto faltante: {item['filename']}")
    blob_reload_counted(blob)
    remote_sha = read_remote_sha256(blob)
    if int(blob.size) != int(item["sizeBytes"]) or remote_sha != item["sha256"]:
        raise RuntimeError(f"Objeto remoto incompatible: {item['filename']}")
    objects_existing_verified += 1
    objects_remote_verified += 1
    return {
        "filename": item["filename"],
        "status": "existing_verified",
        "generation": blob.generation,
        "etag": blob.etag,
    }


def verify_remote_receipt(bucket, expected_sha256: str | None = None) -> tuple[dict[str, Any], str]:
    receipt_blob = bucket.blob(f"{CFG.destination_prefix}/publish_receipt.json")
    if not blob_exists_counted(receipt_blob):
        raise RuntimeError("Falta publish_receipt.json remoto.")
    receipt_payload, receipt_sha256 = read_remote_json_and_sha256(receipt_blob)
    if expected_sha256 is not None and receipt_sha256 != expected_sha256:
        raise RuntimeError("SHA real de publish_receipt.json no coincide con _SUCCESS.json.")
    if receipt_payload.get("releaseId") != CFG.release_id:
        raise RuntimeError("Receipt remoto pertenece a otra release.")
    if receipt_payload.get("releaseContentSha256") != release_content_sha256:
        raise RuntimeError("Receipt remoto corresponde a otro contenido.")
    if receipt_payload.get("remoteVerificationPassed") is not True:
        raise RuntimeError("Receipt remoto no confirma verificacion remota.")
    return receipt_payload, receipt_sha256


def verify_existing_success_release(bucket) -> tuple[bool, list[dict[str, Any]]]:
    success_blob = bucket.blob(f"{CFG.destination_prefix}/_SUCCESS.json")
    if not blob_exists_counted(success_blob):
        return False, []
    success_payload, _success_sha256 = read_remote_json_and_sha256(success_blob)
    if success_payload.get("releaseId") != CFG.release_id:
        raise RuntimeError("_SUCCESS.json pertenece a otra release.")
    if success_payload.get("status") != "published_and_verified":
        raise RuntimeError("_SUCCESS.json tiene status inesperado.")
    if success_payload.get("releaseContentSha256") != release_content_sha256:
        raise RuntimeError("_SUCCESS.json corresponde a otro contenido.")
    if success_payload.get("remoteVerificationPassed") is not True:
        raise RuntimeError("_SUCCESS.json no confirma verificacion remota.")
    if success_payload.get("publicationArtifactCount") != len(publication_artifacts):
        raise RuntimeError("_SUCCESS.json tiene publicationArtifactCount inesperado.")
    expected_manifest_sha256 = sha256_stream(release_manifest_path)
    if success_payload.get("releaseManifestSha256") != expected_manifest_sha256:
        raise RuntimeError("_SUCCESS.json tiene releaseManifestSha256 inesperado.")
    receipt_sha256 = success_payload.get("receiptSha256")
    if not isinstance(receipt_sha256, str) or not receipt_sha256:
        raise RuntimeError("_SUCCESS.json no contiene receiptSha256 valido.")
    verify_remote_receipt(bucket, expected_sha256=receipt_sha256)
    verified = [verify_remote_artifact(bucket, item) for item in publication_artifacts]
    return True, verified


def write_remote_artifact(bucket, item: dict[str, Any]) -> dict[str, Any]:
    global objects_remote_verified
    source_path = Path(item["sourcePath"])
    blob = bucket.blob(item["destinationObject"])
    if blob_exists_counted(blob):
        return verify_remote_artifact(bucket, item)
    blob.metadata = item["metadata"]
    blob.content_type = item["contentType"]
    with source_path.open("rb") as src, blob.open("wb", if_generation_match=0, checksum="auto") as dst:
        shutil.copyfileobj(src, dst, length=1024 * 1024)
    count_gcs_write(item["sizeBytes"])
    blob_reload_counted(blob)
    remote_sha = read_remote_sha256(blob)
    if int(blob.size) != int(item["sizeBytes"]) or remote_sha != item["sha256"]:
        raise RuntimeError(f"Verificacion remota fallo para {item['filename']}")
    objects_remote_verified += 1
    return {
        "filename": item["filename"],
        "status": "uploaded_verified",
        "generation": blob.generation,
        "etag": blob.etag,
    }


def ensure_publish_receipt(bucket, verified_artifacts: list[dict[str, Any]]) -> tuple[str, dict[str, Any]]:
    receipt_blob = bucket.blob(f"{CFG.destination_prefix}/publish_receipt.json")
    if blob_exists_counted(receipt_blob):
        _receipt_payload, receipt_sha256 = verify_remote_receipt(bucket)
        blob_reload_counted(receipt_blob)
        return receipt_sha256, {
            "filename": "publish_receipt.json",
            "status": "existing_receipt_reused",
            "generation": receipt_blob.generation,
            "etag": receipt_blob.etag,
        }

    receipt = {
        "releaseId": CFG.release_id,
        "releaseContentSha256": release_content_sha256,
        "releaseManifestSha256": sha256_stream(release_manifest_path),
        "releasedAtUtc": datetime.now(timezone.utc).isoformat(),
        "publicationArtifactCount": len(publication_artifacts),
        "artifacts": publication_artifacts,
        "verificationResults": verified_artifacts,
        "remoteVerificationPassed": len(verified_artifacts) == len(publication_artifacts),
    }
    if receipt["remoteVerificationPassed"] is not True:
        raise RuntimeError("No se puede emitir receipt sin verificacion remota completa.")
    receipt_path = CFG.staging_dir / "publish_receipt.json"
    atomic_write_json(receipt_path, receipt)
    receipt_entry = artifact_entry({
        "filename": "publish_receipt.json",
        "source": receipt_path,
        "destination": f"gs://{CFG.bucket_name}/{CFG.destination_prefix}/publish_receipt.json",
        "content_type": "application/json",
    })
    receipt_result = write_remote_artifact(bucket, receipt_entry)
    return sha256_stream(receipt_path), receipt_result


def synthetic_publication_idempotence_self_check() -> dict[str, Any]:
    def evaluate_success(success_payload: dict[str, Any], receipt_sha256: str, artifact_count: int) -> dict[str, Any]:
        if success_payload.get("receiptSha256") != receipt_sha256:
            raise RuntimeError("success incompatible")
        if success_payload.get("publicationArtifactCount") != artifact_count:
            raise RuntimeError("success incompatible")
        return {"status": "existing_release_verified", "gcs_write_operations": 0}

    def evaluate_receipt(receipt_payload: dict[str, Any], release_id: str, release_sha256: str) -> dict[str, Any]:
        if receipt_payload.get("releaseId") != release_id:
            raise RuntimeError("receipt incompatible")
        if receipt_payload.get("releaseContentSha256") != release_sha256:
            raise RuntimeError("receipt incompatible")
        if receipt_payload.get("remoteVerificationPassed") is not True:
            raise RuntimeError("receipt incompatible")
        return {"receipt_reused": True, "gcs_write_operations": 0}

    compatible_success = {"receiptSha256": "abc", "publicationArtifactCount": 3}
    compatible_receipt = {
        "releaseId": "rel",
        "releaseContentSha256": "content",
        "remoteVerificationPassed": True,
    }
    success_case = evaluate_success(compatible_success, "abc", 3)
    receipt_case = evaluate_receipt(compatible_receipt, "rel", "content")
    receipt_error = False
    success_error = False
    try:
        evaluate_receipt({**compatible_receipt, "remoteVerificationPassed": False}, "rel", "content")
    except RuntimeError:
        receipt_error = True
    try:
        evaluate_success({**compatible_success, "receiptSha256": "other"}, "abc", 3)
    except RuntimeError:
        success_error = True
    return {
        "existing_success_zero_writes": success_case["gcs_write_operations"] == 0,
        "existing_receipt_reused": receipt_case["receipt_reused"] is True,
        "incompatible_receipt_errors": receipt_error,
        "incompatible_success_errors": success_error,
    }


client, bucket = authenticate_for_publication()
publication_results: list[dict[str, Any]] = []
success_marker_exists = False
release_success = False
publication_status = "dry_run_ready"

if bucket is not None:
    if CFG.destination_prefix != "models/releases/sagittal_spider_final_v1":
        raise RuntimeError("Prefijo de destino inesperado para publish final v1.")
    for item in publication_artifacts:
        if not item["destinationObject"].startswith(f"{CFG.destination_prefix}/"):
            raise RuntimeError("Artifact fuera del prefijo permitido.")

    existing_success, existing_results = verify_existing_success_release(bucket)
    if existing_success:
        publication_results = existing_results
        publication_status = "existing_release_verified"
        success_marker_exists = True
        release_success = True
    else:
        for item in publication_artifacts:
            publication_results.append(write_remote_artifact(bucket, item))
        receipt_sha256, receipt_result = ensure_publish_receipt(bucket, publication_results)
        publication_results.append(receipt_result)

        success_payload = {
            "releaseId": CFG.release_id,
            "status": "published_and_verified",
            "modelKey": CFG.model_key,
            "releaseContentSha256": release_content_sha256,
            "releaseManifestSha256": sha256_stream(release_manifest_path),
            "receiptSha256": receipt_sha256,
            "artifactCount": len(content_artifacts),
            "publicationArtifactCount": len(publication_artifacts),
            "publishedAtUtc": datetime.now(timezone.utc).isoformat(),
            "remoteVerificationPassed": True,
            "gcsWriteOperations": gcs_write_operations,
        }
        success_path = CFG.staging_dir / "_SUCCESS.json"
        atomic_write_json(success_path, success_payload)
        success_entry = artifact_entry({
            "filename": "_SUCCESS.json",
            "source": success_path,
            "destination": f"gs://{CFG.bucket_name}/{CFG.destination_prefix}/_SUCCESS.json",
            "content_type": "application/json",
        })
        success_result = write_remote_artifact(bucket, success_entry)
        publication_results.append(success_result)
        success_marker_exists = True
        release_success = True
        publication_status = "published_and_verified"

final_json = {
    "DRY_RUN": bucket is None,
    "releaseId": CFG.release_id,
    "publication_status": publication_status,
    "releaseContentSha256": release_content_sha256,
    "releaseManifestSha256": sha256_stream(release_manifest_path),
    "publicationArtifactCount": len(publication_artifacts),
    "publicationResults": publication_results,
    "successMarkerExists": success_marker_exists,
    "releaseSuccess": release_success,
    "gcs_write_operations": gcs_write_operations,
    "gcs_read_operations": gcs_read_operations,
    "bytes_uploaded": bytes_uploaded,
    "objects_existing_verified": objects_existing_verified,
    "objects_remote_verified": objects_remote_verified,
}
print(json.dumps(final_json, indent=2, ensure_ascii=False))